# ⚡ Power-Line Fault Section Identification Using Voltage–Current Waveforms and a 1D CNN

A 500×2 voltage/current transient is classified into one of four 5 km inspection sections on a simulated 20 km line.

> Educational synthetic locator only—not protection equipment or a trip command.

👉 **Open the interactive companion:** [https://power-line-fault-location.streamlit.app](https://power-line-fault-location.streamlit.app/?stage=start)

## Complete workflow

Generate fault waveforms → normalize two channels → train 1D CNN → predict section probabilities → display kilometre region → audit neighbouring errors.

## Interactive learning journey

- [A Line Has Tripped](https://power-line-fault-location.streamlit.app/?stage=problem) — Fault-Location Objective
- [Dividing the 20 km Line](https://power-line-fault-location.streamlit.app/?stage=sections) — Classification Labels
- [Voltage Collapse and Current Rise](https://power-line-fault-location.streamlit.app/?stage=physics) — Waveform Evidence
- [Simulated Fault Tests](https://power-line-fault-location.streamlit.app/?stage=generate) — Synthetic Training Data
- [Preparing Relay Records](https://power-line-fault-location.streamlit.app/?stage=prepare) — Signal Normalization
- [Scanning the Electrical Transient](https://power-line-fault-location.streamlit.app/?stage=cnn) — 1D Convolutional Neural Network
- [Learning From Labelled Faults](https://power-line-fault-location.streamlit.app/?stage=training) — Supervised Training
- [Where Should the Crew Inspect?](https://power-line-fault-location.streamlit.app/?stage=prediction) — Section Probability
- [The Protection Engineering Audit](https://power-line-fault-location.streamlit.app/?stage=audit) — Confusion Matrix and Baseline

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Conv1D,MaxPooling1D,GlobalAveragePooling1D,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
N=500;SECTIONS=["0–5 km","5–10 km","10–15 km","15–20 km"]

---
# 1. A Line Has Tripped
### Phase 1 of 6 · The Tripped Line

## Part 1 · On the power line
Protection equipment may trip a 20 km line quickly, yet field crews still need a probable location before beginning inspection.

## Part 2 · The engineering challenge
Searching the full route delays restoration, especially where access is difficult or the line crosses multiple terrain zones.

## Part 3 · Where the AI comes in
Classify the short voltage-current transient into one of four 5 km sections to prioritize inspection.

**Electrical Engineering:** A Line Has Tripped → **AI:** Fault-Location Objective → `which 5 km section should be inspected first?`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=problem](https://power-line-fault-location.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

A line trip protects equipment, but restoration work also needs a search region. This model ranks one of four sections for inspection. It does not replace impedance-based protection, travelling-wave methods, or field procedures.

## Part 5 · What you just built

**In the notebook:** Define the 20 km line and four output classes.

**Takeaway:** The output is an inspection region, not an exact repair location.

[Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Dividing the 20 km Line](https://power-line-fault-location.streamlit.app/?stage=sections) ▶

---
# 2. Dividing the 20 km Line
### Phase 1 of 6 · The Tripped Line

## Part 1 · On the power line
The line is divided into four operational search regions measured from Substation A.

## Part 2 · The engineering challenge
A continuous kilometre estimate can imply more precision than a simplified synthetic model supports.

## Part 3 · Where the AI comes in
Assign every simulated fault distance to a transparent section label and train a four-class classifier.

**Electrical Engineering:** Dividing the 20 km Line → **AI:** Classification Labels → `0-5, 5-10, 10-15, 15-20 km`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=sections](https://power-line-fault-location.streamlit.app/?stage=sections)

## Part 4 · The technical explanation

In [ ]:
def section_label(distance_km):return min(3,int(distance_km//5))
for d in [2,7,12,17]:print(d,"km → Section",section_label(d)+1,SECTIONS[section_label(d)])

## Part 5 · What you just built

**In the notebook:** Convert fault distance into Section 1–4 labels.

**Takeaway:** Section classification keeps the educational claim aligned with model resolution.

◀ [Previous: A Line Has Tripped](https://power-line-fault-location.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Voltage Collapse and Current Rise](https://power-line-fault-location.streamlit.app/?stage=physics) ▶

---
# 3. Voltage Collapse and Current Rise
### Phase 2 of 6 · Electrical Evidence

## Part 1 · On the power line
A short circuit depresses voltage and increases current. Line impedance between the source and fault changes the magnitude and transient response with distance.

## Part 2 · The engineering challenge
Fault resistance, source strength, inception angle, and noise can make simple magnitude rules ambiguous.

## Part 3 · Where the AI comes in
Use the complete synchronized waveform so the model can combine drop, spike, oscillation, timing, and recovery shape.

**Electrical Engineering:** Voltage Collapse and Current Rise → **AI:** Waveform Evidence → `distance, impedance, resistance alter transient shape`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=physics](https://power-line-fault-location.streamlit.app/?stage=physics)

## Part 4 · The technical explanation

In [ ]:
def simulate_fault(distance,resistance,seed=0):
 rng=np.random.default_rng(seed);t=np.linspace(-.025,.075,N);phase=2*np.pi*50*t;v=np.sin(phase);i=.55*np.sin(phase-.18);on=t>=0
 attenuation=.34+.025*distance+.13*resistance;surge=5.2-.13*distance-.8*resistance
 v[on]*=attenuation;i[on]=surge*np.sin(phase[on]-.35)+.9*np.exp(-t[on]/.012)*np.sin(2*np.pi*520*t[on])
 v+=rng.normal(0,.025,N);i+=rng.normal(0,.045,N);return t,np.stack([v,i],axis=1).astype("float32")
t,w=simulate_fault(12.5,.3,7)
plt.figure(figsize=(13,4));plt.plot(t*1000,w[:,0],label="Voltage");plt.plot(t*1000,w[:,1],label="Current");plt.axvline(0,color="black",ls="--");plt.xlabel("Time (ms)");plt.ylabel("pu");plt.grid(alpha=.2);plt.legend();plt.show()

## Part 5 · What you just built

**In the notebook:** Plot normal and faulted voltage/current traces.

**Takeaway:** Distance information is distributed across the transient, not stored in one sample.

◀ [Previous: Dividing the 20 km Line](https://power-line-fault-location.streamlit.app/?stage=sections) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Simulated Fault Tests](https://power-line-fault-location.streamlit.app/?stage=generate) ▶

---
# 4. Simulated Fault Tests
### Phase 2 of 6 · Electrical Evidence

## Part 1 · On the power line
Controlled studies vary fault distance, resistance, source strength, inception angle, and measurement noise.

## Part 2 · The engineering challenge
A classroom notebook needs many labelled examples but cannot create real transmission-line faults.

## Part 3 · Where the AI comes in
Generate simplified physically motivated waveforms and preserve the continuous distance as metadata only.

**Electrical Engineering:** Simulated Fault Tests → **AI:** Synthetic Training Data → `500 samples × 2 channels`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=generate](https://power-line-fault-location.streamlit.app/?stage=generate)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);X,y,distances=[],[],[]
for label in range(4):
 for k in range(450):
  d=rng.uniform(label*5+.05,(label+1)*5-.05);r=rng.uniform(0,1);_,wave=simulate_fault(d,r,10000*label+k);X.append(wave);y.append(label);distances.append(d)
X=np.array(X);y=np.array(y);distances=np.array(distances)
print(X.shape,"= examples × 500 samples × 2 channels")
pd.DataFrame({"distance_km":distances[:8],"label":y[:8],"voltage_min":X[:8,:,0].min(1),"current_peak":np.abs(X[:8,:,1]).max(1)})

## Part 5 · What you just built

**In the notebook:** Create balanced examples across 0–20 km with voltage and current channels.

**Takeaway:** Synthetic data teaches the workflow but does not validate a field protection system.

◀ [Previous: Voltage Collapse and Current Rise](https://power-line-fault-location.streamlit.app/?stage=physics) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Preparing Relay Records](https://power-line-fault-location.streamlit.app/?stage=prepare) ▶

---
# 5. Preparing Relay Records
### Phase 3 of 6 · Preparing Signals

## Part 1 · On the power line
Voltage and current use different per-unit ranges, and every record must align around the disturbance.

## Part 2 · The engineering challenge
Scaling with test data leaks future information; scaling each record independently can erase useful magnitude differences.

## Part 3 · Where the AI comes in
Align 500 samples and normalize each channel using training-set statistics only.

**Electrical Engineering:** Preparing Relay Records → **AI:** Signal Normalization → `per-channel training statistics`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=prepare](https://power-line-fault-location.streamlit.app/?stage=prepare)

## Part 4 · The technical explanation

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=SEED)
X_train,X_val,y_train,y_val=train_test_split(X_train,y_train,test_size=.20,stratify=y_train,random_state=SEED)
scaler=StandardScaler().fit(X_train.reshape(-1,2))
def scale(a):return scaler.transform(a.reshape(-1,2)).reshape(a.shape)
X_train,X_val,X_test=scale(X_train),scale(X_val),scale(X_test)
print(X_train.shape,X_val.shape,X_test.shape)

## Part 5 · What you just built

**In the notebook:** Create stratified splits and transform two channels without leakage.

**Takeaway:** Signal preparation must preserve the distance-dependent evidence the model needs.

◀ [Previous: Simulated Fault Tests](https://power-line-fault-location.streamlit.app/?stage=generate) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Scanning the Electrical Transient](https://power-line-fault-location.streamlit.app/?stage=cnn) ▶

---
# 6. Scanning the Electrical Transient
### Phase 4 of 6 · Learning the Transient

## Part 1 · On the power line
Engineers inspect local events: sudden voltage reduction, current surge, damped oscillation, and recovery.

## Part 2 · The engineering challenge
Hand-written thresholds cannot easily combine many local patterns under varying fault conditions.

## Part 3 · Where the AI comes in
One-dimensional filters slide directly along time across both channels and learn useful transient motifs.

**Electrical Engineering:** Scanning the Electrical Transient → **AI:** 1D Convolutional Neural Network → `Conv1D -> pool -> Conv1D -> global average`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=cnn](https://power-line-fault-location.streamlit.app/?stage=cnn)

## Part 4 · The technical explanation

In [ ]:
model=Sequential([Input((500,2)),Conv1D(32,9,activation="relu"),MaxPooling1D(2),Conv1D(64,7,activation="relu"),MaxPooling1D(2),Conv1D(96,5,activation="relu"),GlobalAveragePooling1D(),Dropout(.2),Dense(32,activation="relu"),Dense(4,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"]);model.summary()

## Part 5 · What you just built

**In the notebook:** Build a compact Conv1D classifier ending in four Softmax outputs.

**Takeaway:** A 1D CNN matches time-domain waveform structure without converting signals into images.

◀ [Previous: Preparing Relay Records](https://power-line-fault-location.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning From Labelled Faults](https://power-line-fault-location.streamlit.app/?stage=training) ▶

---
# 7. Learning From Labelled Faults
### Phase 4 of 6 · Learning the Transient

## Part 1 · On the power line
Each simulated record has a known fault distance and therefore a known section.

## Part 2 · The engineering challenge
An apparently high score can come from imbalance or leakage between nearly identical simulations.

## Part 3 · Where the AI comes in
Use balanced sections, stratified splits, validation monitoring, and an untouched test set.

**Electrical Engineering:** Learning From Labelled Faults → **AI:** Supervised Training → `cross-entropy + Adam + early stopping`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=training](https://power-line-fault-location.streamlit.app/?stage=training)

## Part 4 · The technical explanation

In [ ]:
early=EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True)
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=45,batch_size=48,callbacks=[early],verbose=0)
fig,ax=plt.subplots(1,2,figsize=(12,4));ax[0].plot(history.history["loss"],label="train");ax[0].plot(history.history["val_loss"],label="validation");ax[1].plot(history.history["accuracy"],label="train");ax[1].plot(history.history["val_accuracy"],label="validation")
for a in ax:a.set_xlabel("Epoch");a.grid(alpha=.2);a.legend()
plt.show()

## Part 5 · What you just built

**In the notebook:** Train the CNN and plot loss and accuracy curves.

**Takeaway:** Testing must use waveforms the CNN did not optimize against.

◀ [Previous: Scanning the Electrical Transient](https://power-line-fault-location.streamlit.app/?stage=cnn) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Where Should the Crew Inspect?](https://power-line-fault-location.streamlit.app/?stage=prediction) ▶

---
# 8. Where Should the Crew Inspect?
### Phase 5 of 6 · Locating the Section

## Part 1 · On the power line
The control room needs a probable region expressed in kilometres from Substation A.

## Part 2 · The engineering challenge
The largest probability can still be uncertain, especially close to a section boundary.

## Part 3 · Where the AI comes in
Report all section probabilities, confidence, and the kilometre interval; flag low-confidence cases for wider inspection.

**Electrical Engineering:** Where Should the Crew Inspect? → **AI:** Section Probability → `Softmax -> Section 1–4`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=prediction](https://power-line-fault-location.streamlit.app/?stage=prediction)

## Part 4 · The technical explanation

In [ ]:
distance=12.5;_,new_wave=simulate_fault(distance,.3,999);new_scaled=scale(new_wave[None,...]);probs=model.predict(new_scaled,verbose=0)[0];pred=int(np.argmax(probs))
print("FAULT DETECTED");print("Predicted fault section: SECTION",pred+1);print("Region:",SECTIONS[pred],"from Substation A");print("Confidence:",f"{probs[pred]:.1%}")
for i,p in enumerate(probs):print(f"Section {i+1}: {p:.1%}")

## Part 5 · What you just built

**In the notebook:** Predict a new waveform and display the probable section and region.

**Takeaway:** Confidence communicates uncertainty; it does not guarantee the fault lies inside the interval.

◀ [Previous: Learning From Labelled Faults](https://power-line-fault-location.streamlit.app/?stage=training) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Protection Engineering Audit](https://power-line-fault-location.streamlit.app/?stage=audit) ▶

---
# 9. The Protection Engineering Audit
### Phase 6 of 6 · Protection Audit

## Part 1 · On the power line
A useful locator should reduce search time without misleading crews or interfering with primary protection.

## Part 2 · The engineering challenge
Overall accuracy hides whether mistakes jump across the line or occur mainly between neighbouring sections.

## Part 3 · Where the AI comes in
Compare with a simple magnitude baseline, inspect the confusion matrix, and count adjacent versus non-adjacent errors.

**Electrical Engineering:** The Protection Engineering Audit → **AI:** Confusion Matrix and Baseline → `CNN vs magnitude threshold; neighbour errors`

> 🎬 **See this illustrated and interactive:** [https://power-line-fault-location.streamlit.app/?stage=audit](https://power-line-fault-location.streamlit.app/?stage=audit)

## Part 4 · The technical explanation

In [ ]:
start=time.perf_counter();test_probs=model.predict(X_test,verbose=0);elapsed=time.perf_counter()-start;pred=np.argmax(test_probs,axis=1)
print(classification_report(y_test,pred,target_names=[f"Section {i}" for i in range(1,5)]));print(f"Mean inference time: {1000*elapsed/len(X_test):.3f} ms/sample")
errors=np.abs(pred-y_test);print("Adjacent-section errors:",int((errors==1).sum()));print("Non-adjacent errors:",int((errors>1).sum()))
ConfusionMatrixDisplay(confusion_matrix(y_test,pred),display_labels=["S1","S2","S3","S4"]).plot(cmap="Blues");plt.show()
# Deliberately weak magnitude-only baseline.
peak=np.abs(X_test[:,:,1]).max(1);threshold_pred=np.digitize(peak,np.quantile(peak,[.25,.5,.75]));print("Magnitude baseline accuracy:",accuracy_score(y_test,threshold_pred));print("1D CNN accuracy:",accuracy_score(y_test,pred))
print("Field exclusions: simplified line, synthetic source/fault model, no CT/CVT saturation, no topology changes, no timing uncertainty, no protection coordination study.")

## Part 5 · What you just built

**In the notebook:** Report test metrics, inference time, confusion matrix, and field-validation exclusions.

**Takeaway:** This model prioritizes inspection; it is not a relay, trip command, or field-validated locator.

◀ [Previous: Where Should the Crew Inspect?](https://power-line-fault-location.streamlit.app/?stage=prediction) &nbsp;|&nbsp; [Project overview](https://power-line-fault-location.streamlit.app/?stage=start)

---
# Final engineering conclusion

The 1D CNN learns directly from synchronized voltage and current transients and returns a probable 5 km inspection region. The output can prioritize field inspection, but the synthetic notebook is not protection logic or a validated utility locator.